<a href="https://colab.research.google.com/github/telyotarsyn/simple_llm_rag/blob/main/Simple_RAG_petproject.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!nvidia-smi

Mon Sep 14 18:41:19 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   48C    P8             14W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# Download PDF file
import os
import requests

# Get PDF document
pdf_path = "dune.pdf"

# Download PDF if it doesn't already exist
if not os.path.exists(pdf_path):
  print("File doesn't exist, downloading...")

  # The URL of the PDF you want to download
  url = "https://www.hetako.ee/kamp/ulme/Frank%20Herbert%20-%20Dune%201%20-%20Dune.pdf"

  # The local filename to save the downloaded file
  filename = pdf_path

  # Send a GET request to the URL
  response = requests.get(url)

  # Check if the request was successful
  if response.status_code == 200:
      # Open a file in binary write mode and save the content to it
      with open(filename, "wb") as file:
          file.write(response.content)
      print(f"The file has been downloaded and saved as {filename}")
  else:
      print(f"Failed to download the file. Status code: {response.status_code}")
else:
  print(f"File {pdf_path} exists.")

File doesn't exist, downloading...
The file has been downloaded and saved as dune.pdf


In [3]:
# Perform Google Colab installs (if running in Google Colab)
import os

if "COLAB_GPU" in os.environ:
    print("[INFO] Running in Google Colab, installing requirements.")
    !pip install -U torch # requires torch 2.1.1+ (for efficient sdpa implementation)
    !pip install PyMuPDF # for reading PDFs with Python
    !pip install tqdm # for progress bars
    !pip install sentence-transformers # for embedding models
    !pip install accelerate # for quantization model loading
    !pip install bitsandbytes # for quantizing models (less storage space)

[INFO] Running in Google Colab, installing requirements.


In [9]:
    !pip install flash-attn --no-build-isolation # for faster attention mechanism = faster LLM inference

  Using cached flash_attn-2.8.3.post1.tar.gz (8.5 MB)
  Preparing metadata (setup.py) ... done
  error: subprocess-exited-with-error
  
  × python setup.py bdist_wheel did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for flash-attn
  Running setup.py clean for flash-attn
Failed to build flash-attn
ERROR: ERROR: Failed to build installable wheels for some pyproject.toml based projects (flash-attn)


In [10]:
# Requires !pip install PyMuPDF, see: https://github.com/pymupdf/pymupdf
import fitz # (pymupdf, found this is better than pypdf for our use case, note: licence is AGPL-3.0, keep that in mind if you want to use any code commercially)
from tqdm.auto import tqdm # for progress bars, requires !pip install tqdm

def text_formatter(text: str) -> str:
    """Performs minor formatting on text."""
    cleaned_text = text.replace("\n", " ").strip() # note: this might be different for each doc (best to experiment)

    # Other potential text formatting functions can go here
    return cleaned_text

# Open PDF and get lines/pages
# Note: this only focuses on text, rather than images/figures etc
def open_and_read_pdf(pdf_path: str) -> list[dict]:
    """
    Opens a PDF file, reads its text content page by page, and collects statistics.

    Parameters:
        pdf_path (str): The file path to the PDF document to be opened and read.

    Returns:
        list[dict]: A list of dictionaries, each containing the page number
        (adjusted), character count, word count, sentence count, token count, and the extracted text
        for each page.
    """
    doc = fitz.open(pdf_path)  # open a document
    pages_and_texts = []
    for page_number, page in tqdm(enumerate(doc)):  # iterate the document pages
        text = page.get_text()  # get plain text encoded as UTF-8
        text = text_formatter(text)
        pages_and_texts.append({"page_number": page_number - 41,  # adjust page numbers since our PDF starts on page 42
                                "page_char_count": len(text),
                                "page_word_count": len(text.split(" ")),
                                "page_sentence_count_raw": len(text.split(". ")),
                                "page_token_count": len(text) / 4,  # 1 token = ~4 chars, see: https://help.openai.com/en/articles/4936856-what-are-tokens-and-how-to-count-them
                                "text": text})
    return pages_and_texts

pages_and_texts = open_and_read_pdf(pdf_path=pdf_path)
pages_and_texts[:2]

0it [00:00, ?it/s]

[{'page_number': -41,
  'page_char_count': 119,
  'page_word_count': 29,
  'page_sentence_count_raw': 1,
  'page_token_count': 29.75,
  'text': 'Converted to  Converted to  Converted to  Converted to “PDF PDF PDF PDF” by  ” by  ” by  ” by ->MKM< >MKM< >MKM< >MKM<-'},
 {'page_number': -40,
  'page_char_count': 2291,
  'page_word_count': 498,
  'page_sentence_count_raw': 24,
  'page_token_count': 572.75,
  'text': 'Dune  Frank Herbert    Copyright 1965    Book 1  DUNE    = = = = = =     A beginning is the time for taking the most delicate care that the balances are  correct. This every sister of the Bene Gesserit knows. To begin your study of  the life of Muad\'Dib, then, take care that you first place him in his time: born  in the 57th year of the Padishah Emperor, Shaddam IV. And take the most special  care that you locate Muad\'Dib in his place: the planet Arrakis. Do not be  deceived by the fact that he was born on Caladan and lived his first fifteen  years there. Arrakis, the planet

In [11]:
import random

random.sample(pages_and_texts, k=3)

[{'page_number': 250,
  'page_char_count': 3687,
  'page_word_count': 762,
  'page_sentence_count_raw': 40,
  'page_token_count': 921.75,
  'text': 'ship glistened in the flat light of the sun, but the shadow side still showed  yellow portholes from glowglobes of the night. Beyond the ship, the city of  Arrakeen lay cold and gleaming in the light of the northern sun.      It wasn\'t the lighter that excited Stilgar\'s awe, Paul knew, but the  construction for which the lighter was only the centerpost. A single metal  hutment, many stories tall, reached out in a thousand-meter circle from the base  of the lighter -- a tent composed of interlocking metal leaves -- the temporary  lodging place for five legions of Sardaukar and His Imperial Majesty, the  Padishah Emperor Shaddam IV.      From his position squatting at Paul\'s left, Gurney Halleck said: "I count  nine levels to it. Must be quite a few Sardaukar in there."       "Five legions," Paul said.      "It grows light," Stilgar hisse

In [12]:
import pandas as pd

df = pd.DataFrame(pages_and_texts)
df.head()

,page_number,page_char_count,page_word_count,page_sentence_count_raw,page_token_count,text
0,-41,119,29,1,29.75,Converted to Converted to Converted to Conv...
1,-40,2291,498,24,572.75,Dune Frank Herbert Copyright 1965 Book ...
2,-39,3591,736,79,897.75,"""Sleep well, you sly little rascal,"" said the ..."
3,-38,3562,736,44,890.50,When dawn touched Paul's window sill with yell...
4,-37,3513,788,63,878.25,"Now, there was a man who appreciated the power..."


In [13]:
# Get stats
df.describe().round(2)

,page_number,page_char_count,page_word_count,page_sentence_count_raw,page_token_count
count,345.00,345.00,345.00,345.00,345.00
mean,131.00,3500.50,740.63,46.57,875.13
std,99.74,364.47,79.78,12.04,91.12
min,-41.00,0.00,1.00,1.00,0.00
25%,45.00,3406.00,727.00,40.00,851.50
50%,131.00,3536.00,754.00,46.00,884.00
75%,217.00,3668.00,777.00,51.00,917.00
max,303.00,4169.00,837.00,130.00,1042.25


In [14]:
from spacy.lang.en import English # see https://spacy.io/usage for install instructions

nlp = English()

# Add a sentencizer pipeline, see https://spacy.io/api/sentencizer/
nlp.add_pipe("sentencizer")

# Create a document instance as an example
doc = nlp("This is a sentence. This another sentence.")
assert len(list(doc.sents)) == 2

# Access the sentences of the document
list(doc.sents)

[This is a sentence., This another sentence.]